In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import warnings
warnings.filterwarnings('ignore')

class TrendClusterClassifier:
    def __init__(self, n_top_features=3, n_clusters=3, window_size=3):
        self.trend_features = {}
        self.cluster_classifiers = {}
        self.congestion_classifier = None
        self.scaler = StandardScaler()
        self.n_top_features = n_top_features
        self.n_clusters = n_clusters
        self.window_size = window_size
        self.congestion_mappings = {}
        self.label_encoder = LabelEncoder()
        self.imputer = SimpleImputer(strategy='median')
        self.feature_encoders = {}
        self.aggregated_trend_features = []
        
    def _preprocess_data(self, X, is_training=False):
        X_processed = X.copy()
        
        categorical_cols = X_processed.select_dtypes(include=['object', 'category']).columns
        
        for col in categorical_cols:
            if is_training:
                le = LabelEncoder()
                X_processed[col] = le.fit_transform(X_processed[col].astype(str))
                self.feature_encoders[col] = le
            else:
                if col in self.feature_encoders:
                    le = self.feature_encoders[col]
                    mask = ~X_processed[col].isin(le.classes_)
                    if mask.any():
                        X_processed.loc[mask, col] = le.classes_[0]
                    X_processed[col] = le.transform(X_processed[col].astype(str))
                else:
                    X_processed[col] = pd.factorize(X_processed[col])[0]
        
        if len(X_processed) > 0:
            if is_training:
                X_processed = pd.DataFrame(
                    self.imputer.fit_transform(X_processed),
                    columns=X_processed.columns
                )
            else:
                X_processed = pd.DataFrame(
                    self.imputer.transform(X_processed),
                    columns=X_processed.columns
                )
        
        return X_processed
    
    def _calculate_sliding_window_trend(self, feature_values):
        if len(feature_values) < self.window_size:
            return np.zeros(len(feature_values))
        
        trend_features = np.zeros(len(feature_values))
        
        for i in range(len(feature_values)):
            start_idx = max(0, i - self.window_size + 1)
            end_idx = i + 1
            
            window_data = feature_values[start_idx:end_idx]
            
            if len(window_data) < 2:
                trend_features[i] = 0
                continue
            
            try:
                x = np.arange(len(window_data))
                y = np.array(window_data, dtype=np.float64)
                
                mask = ~np.isnan(y)
                if np.sum(mask) < 2:
                    trend_features[i] = 0
                    continue
                
                x_clean = x[mask]
                y_clean = y[mask]
                
                A = np.vstack([x_clean, np.ones(len(x_clean))]).T
                slope, _ = np.linalg.lstsq(A, y_clean, rcond=None)[0]
                
                y_pred = slope * x_clean
                ss_res = np.sum((y_clean - y_pred) ** 2)
                ss_tot = np.sum((y_clean - np.mean(y_clean)) ** 2)
                
                if ss_tot == 0:
                    r_squared = 0
                else:
                    r_squared = 1 - (ss_res / ss_tot)
                
                trend_score = abs(slope) * r_squared
                trend_features[i] = trend_score
                
            except:
                trend_features[i] = 0
        
        return trend_features
    
    def _extract_trend_features(self, X_data):
        trend_features_df = pd.DataFrame(index=X_data.index)
        
        for col in X_data.columns:
            try:
                trend_values = self._calculate_sliding_window_trend(X_data[col].values)
                trend_features_df[f'{col}_trend'] = trend_values
            except:
                trend_features_df[f'{col}_trend'] = np.zeros(len(X_data))
        
        return trend_features_df
    
    def _identify_trend_features(self, X_train, y_train):
        unique_classes = np.unique(y_train)
        self.trend_features = {}
        
        X_trend = self._extract_trend_features(X_train)
        
        for cls in unique_classes:
            class_mask = y_train == cls
            class_data = X_trend[class_mask]
            
            if len(class_data) < 3:
                available_features = X_trend.columns.tolist()[:self.n_top_features]
                self.trend_features[cls] = available_features
                continue
            
            feature_scores = []
            
            for col in X_trend.columns:
                try:
                    trend_strength = np.mean(np.abs(class_data[col].values))
                    feature_scores.append((col, trend_strength))
                except:
                    feature_scores.append((col, 0))
            
            feature_scores.sort(key=lambda x: x[1], reverse=True)
            top_features = [feat for feat, score in feature_scores[:self.n_top_features]]
            self.trend_features[cls] = top_features
        
        self._aggregate_trend_features(X_trend)
    
    def _aggregate_trend_features(self, X_trend):
        all_trend_features = set()
        
        for cls in self.trend_features:
            all_trend_features.update(self.trend_features[cls])
        
        if all_trend_features:
            feature_scores = []
            for feat in all_trend_features:
                if feat in X_trend.columns:
                    overall_strength = np.mean(np.abs(X_trend[feat].values))
                    feature_scores.append((feat, overall_strength))
            
            feature_scores.sort(key=lambda x: x[1], reverse=True)
            self.aggregated_trend_features = [feat for feat, score in feature_scores[:self.n_top_features]]
        else:
            self.aggregated_trend_features = X_trend.columns.tolist()[:self.n_top_features]
    
    def _calculate_congestion(self, X_scaled, cluster_labels, cluster_centers):
        congestion_levels = np.zeros(len(X_scaled), dtype=int)
        
        for i in range(self.n_clusters):
            cluster_points = X_scaled[cluster_labels == i]
            
            if len(cluster_points) < 2:
                congestion_levels[cluster_labels == i] = 1
                continue
            
            centroid = cluster_centers[i]
            distances = np.linalg.norm(cluster_points - centroid, axis=1)
            
            avg_distance = np.mean(distances)
            std_distance = np.std(distances)
            
            congestion_score = 1 / (avg_distance + 1e-10)
            
            if congestion_score > np.percentile([1/(d+1e-10) for d in distances], 75):
                congestion_level = 2
            elif congestion_score > np.percentile([1/(d+1e-10) for d in distances], 25):
                congestion_level = 1
            else:
                congestion_level = 0
            
            congestion_levels[cluster_labels == i] = congestion_level
        
        return congestion_levels
    
    def _cluster_data(self, X_class, trend_features):
        if len(X_class) < self.n_clusters:
            return np.zeros(len(X_class), dtype=int), {}
        
        available_features = [f for f in trend_features if f in X_class.columns]
        if len(available_features) == 0:
            return np.zeros(len(X_class), dtype=int), {}
        
        X_cluster = X_class[available_features].values
        
        if np.isnan(X_cluster).any():
            X_cluster = np.nan_to_num(X_cluster)
        
        if X_cluster.shape[1] == 0:
            return np.zeros(len(X_class), dtype=int), {}
        
        try:
            X_scaled = self.scaler.fit_transform(X_cluster)
        except:
            X_scaled = X_cluster
        
        try:
            kmeans = KMeans(n_clusters=self.n_clusters, random_state=42, n_init=10)
            cluster_labels = kmeans.fit_predict(X_scaled)
            
            congestion_levels = self._calculate_congestion(X_scaled, cluster_labels, kmeans.cluster_centers_)
            
            congestion_mapping = {}
            for i in range(self.n_clusters):
                congestion_mapping[i] = congestion_levels[cluster_labels == i][0] if len(congestion_levels[cluster_labels == i]) > 0 else 1
            
            return cluster_labels, congestion_mapping
            
        except Exception as e:
            return np.zeros(len(X_class), dtype=int), {}
    
    def fit(self, X_train, y_train):
        print("Step 1: Preprocessing data...")
        y_train_encoded = self.label_encoder.fit_transform(y_train)
        X_train_processed = self._preprocess_data(X_train, is_training=True)
        
        print("Step 2: Extracting and identifying trend features...")
        self._identify_trend_features(X_train_processed, y_train_encoded)
        
        print("Step 3: Clustering data and assigning congestion levels...")
        congestion_labels = np.zeros(len(X_train_processed), dtype=int)
        all_congestion_data = []
        all_congestion_labels = []
        
        X_trend = self._extract_trend_features(X_train_processed)
        
        for cls, features in self.trend_features.items():
            class_mask = y_train_encoded == cls
            X_class_trend = X_trend[class_mask]
            
            if len(X_class_trend) < self.n_clusters:
                congestion_labels[class_mask] = 1
                continue
            
            cluster_labels, congestion_mapping = self._cluster_data(X_class_trend, features)
            self.congestion_mappings[cls] = congestion_mapping
            
            class_indices = np.where(class_mask)[0]
            for idx, cluster_id in enumerate(cluster_labels):
                original_idx = class_indices[idx]
                congestion_level = congestion_mapping.get(cluster_id, 1)
                congestion_labels[original_idx] = congestion_level
            
            for idx in class_indices:
                trend_features = X_trend.iloc[idx][self.aggregated_trend_features].values
                all_congestion_data.append(trend_features)
                all_congestion_labels.append(congestion_labels[idx])
        
        X_congestion = np.array(all_congestion_data)
        y_congestion = np.array(all_congestion_labels)
        
        print(f"Step 4: Training congestion classifier... (samples: {len(X_congestion)})")
        self.congestion_classifier = RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight='balanced'
        )
        self.congestion_classifier.fit(X_congestion, y_congestion)
        
        print("Step 5: Training separate classifiers for each congestion level...")
        for congestion in range(self.n_clusters):
            mask = congestion_labels == congestion
            if np.sum(mask) > 5:
                print(f"  Congestion {congestion}: {np.sum(mask)} samples")
                
                clf = RandomForestClassifier(
                    n_estimators=100,
                    random_state=42,
                    class_weight='balanced'
                )
                clf.fit(X_train_processed[mask], y_train_encoded[mask])
                self.cluster_classifiers[congestion] = clf
            else:
                print(f"  Congestion {congestion}: Not enough samples ({np.sum(mask)})")
        
        print("Training completed!")
    
    def predict(self, X_test):
        X_test_processed = self._preprocess_data(X_test, is_training=False)
        X_test_trend = self._extract_trend_features(X_test_processed)
        
        congestion_predictions = np.zeros(len(X_test_processed), dtype=int)
        
        for i in range(len(X_test_processed)):
            trend_features = X_test_trend.iloc[i][self.aggregated_trend_features].values.reshape(1, -1)
            congestion_pred = self.congestion_classifier.predict(trend_features)
            congestion_predictions[i] = congestion_pred[0]
        
        final_predictions = np.zeros(len(X_test_processed), dtype=int)
        
        for congestion in range(self.n_clusters):
            mask = congestion_predictions == congestion
            if np.sum(mask) > 0 and congestion in self.cluster_classifiers:
                X_congestion = X_test_processed[mask]
                preds = self.cluster_classifiers[congestion].predict(X_congestion)
                final_predictions[mask] = preds
        
        final_predictions_decoded = self.label_encoder.inverse_transform(final_predictions)
        
        return final_predictions_decoded, congestion_predictions
    
    def evaluate(self, X_test, y_test):
        y_pred, congestion_pred = self.predict(X_test)
        
        accuracy = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        
        print("\n" + "="*50)
        print("MODEL EVALUATION")
        print("="*50)
        print(f"Overall Accuracy: {accuracy:.4f}")
        print(f"F1-Score (weighted): {f1:.4f}")
        print(f"Precision (weighted): {precision:.4f}")
        print(f"Recall (weighted): {recall:.4f}")
        
        print(f"\nCongestion Distribution:")
        for congestion in range(self.n_clusters):
            count = np.sum(congestion_pred == congestion)
            percentage = (count / len(congestion_pred)) * 100
            print(f"  Congestion {congestion}: {count} samples ({percentage:.1f}%)")
        
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))
        
        return {
            'accuracy': accuracy,
            'f1_score': f1,
            'precision': precision,
            'recall': recall,
            'y_pred': y_pred,
            'congestion_pred': congestion_pred
        }

def main():
    print("Loading data...")
    try:
        #train_df = pd.read_csv('DATASETS/CDR-MLC/scale_5/Long/CDR-MLC-long.csv')
        train_df = pd.read_csv('DATASETS/CDR-MLC/scale_5/Short/level_1.csv')
        test_df = pd.read_csv('DATASETS/CDR-MLC/scale_5/Short/level_3.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please make sure the dataset files exist.")
        return None
    
    print(f"Train data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
    
    if train_df.shape[1] < 2:
        print("Error: Data must have at least 2 columns (features + target)")
        return None
    
    X_train = train_df.iloc[:, :-1]
    y_train = train_df.iloc[:, -1]
    X_test = test_df.iloc[:, :-1]
    y_test = test_df.iloc[:, -1]
    
    print(f"\nNumber of features: {X_train.shape[1]}")
    print(f"Number of classes: {len(np.unique(y_train))}")
    print(f"Classes: {np.unique(y_train)}")
    
    model = TrendClusterClassifier(n_top_features=10, n_clusters=3, window_size=3)
    model.fit(X_train, y_train)
    
    if len(X_test) > 0:
        results = model.evaluate(X_test, y_test)
        return model, results
    else:
        print("No test data available for evaluation.")
        return model, None

if __name__ == "__main__":
    model, results = main()

Loading data...
Train data shape: (1431, 39)
Test data shape: (1573, 39)

Number of features: 38
Number of classes: 5
Classes: ['HTTP' 'SFTP' 'SMTP' 'SSH' 'VIDEO']
Step 1: Preprocessing data...
Step 2: Extracting and identifying trend features...
Step 3: Clustering data and assigning congestion levels...
Step 4: Training congestion classifier... (samples: 1431)
Step 5: Training separate classifiers for each congestion level...
  Congestion 0: 1208 samples
  Congestion 1: 223 samples
  Congestion 2: Not enough samples (0)
Training completed!

MODEL EVALUATION
Overall Accuracy: 0.8633
F1-Score (weighted): 0.8477
Precision (weighted): 0.8729
Recall (weighted): 0.8633

Congestion Distribution:
  Congestion 0: 1357 samples (86.3%)
  Congestion 1: 216 samples (13.7%)
  Congestion 2: 0 samples (0.0%)

Classification Report:
              precision    recall  f1-score   support

        HTTP       0.88      0.99      0.93      1092
        SFTP       1.00      0.32      0.49        37
        

In [3]:

def main():
    print("Loading data...")
    try:
        train_df = pd.read_csv('DATASETS/CDR-MLC/scale_5/Short/level_2.csv')
        test_df = pd.read_csv('DATASETS/CDR-MLC/scale_5/Short/level_3.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please make sure the dataset files exist.")
        return None
    
    print(f"Train data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
    
    if train_df.shape[1] < 2:
        print("Error: Data must have at least 2 columns (features + target)")
        return None
    
    X_train = train_df.iloc[:, :-1]
    y_train = train_df.iloc[:, -1]
    X_test = test_df.iloc[:, :-1]
    y_test = test_df.iloc[:, -1]
    
    print(f"\nNumber of features: {X_train.shape[1]}")
    print(f"Number of classes: {len(np.unique(y_train))}")
    print(f"Classes: {np.unique(y_train)}")
    
    model = TrendClusterClassifier(n_top_features=10, n_clusters=3, window_size=3)
    model.fit(X_train, y_train)
    
    if len(X_test) > 0:
        results = model.evaluate(X_test, y_test)
        return model, results
    else:
        print("No test data available for evaluation.")
        return model, None

if __name__ == "__main__":
    model, results = main()

Loading data...
Train data shape: (1788, 39)
Test data shape: (1573, 39)

Number of features: 38
Number of classes: 5
Classes: ['HTTP' 'SFTP' 'SMTP' 'SSH' 'VIDEO']
Step 1: Preprocessing data...
Step 2: Extracting and identifying trend features...
Step 3: Clustering data and assigning congestion levels...
Step 4: Training congestion classifier... (samples: 1788)
Step 5: Training separate classifiers for each congestion level...
  Congestion 0: 1558 samples
  Congestion 1: 230 samples
  Congestion 2: Not enough samples (0)
Training completed!

MODEL EVALUATION
Overall Accuracy: 0.9873
F1-Score (weighted): 0.9869
Precision (weighted): 0.9873
Recall (weighted): 0.9873

Congestion Distribution:
  Congestion 0: 1441 samples (91.6%)
  Congestion 1: 132 samples (8.4%)
  Congestion 2: 0 samples (0.0%)

Classification Report:
              precision    recall  f1-score   support

        HTTP       0.99      1.00      0.99      1092
        SFTP       1.00      0.86      0.93        37
        S

In [5]:
def main():
    print("Loading data...")
    try:
        train_df = pd.read_csv('DATASETS/CDR-MLC/scale_0.001/Short/CDR-MLC-Shuffle.csv')
        test_df = pd.read_csv('DATASETS/CDR-MLC/scale_0.001/Long/CDR-MLC-Shuffle.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please make sure the dataset files exist.")
        return None
    
    print(f"Train data shape: {train_df.shape}")
    print(f"Test data shape: {test_df.shape}")
    
    if train_df.shape[1] < 2:
        print("Error: Data must have at least 2 columns (features + target)")
        return None
    
    X_train = train_df.iloc[:, :-1]
    y_train = train_df.iloc[:, -1]
    X_test = test_df.iloc[:, :-1]
    y_test = test_df.iloc[:, -1]
    
    print(f"\nNumber of features: {X_train.shape[1]}")
    print(f"Number of classes: {len(np.unique(y_train))}")
    print(f"Classes: {np.unique(y_train)}")
    
    model = TrendClusterClassifier(n_top_features=10, n_clusters=3, window_size=3)
    model.fit(X_train, y_train)
    
    if len(X_test) > 0:
        results = model.evaluate(X_test, y_test)
        return model, results
    else:
        print("No test data available for evaluation.")
        return model, None

if __name__ == "__main__":
    model, results = main()

Loading data...
Train data shape: (329726, 39)
Test data shape: (1417001, 38)

Number of features: 38
Number of classes: 114530
Classes: [     1      2      3 ... 114528 114529 114530]
Step 1: Preprocessing data...
Step 2: Extracting and identifying trend features...
Step 3: Clustering data and assigning congestion levels...


KeyboardInterrupt: 